<a href="https://colab.research.google.com/github/TrajanDS/SFT_Agent_POC/blob/main/2_create_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Train Test Split File

*   Splits raw data into train, val, test, and OOS sests
*   Creates new project in specified directory
*   {base_dir}/{projects}/{project_name}/{project_[1,2,3,etc]}/split_data/




# input args:

*   tts_dict: train test split percentages (train - float, validation - float test - float, out of time period - date [optional])
• base_dir: - str path to base projects directory where data will be saved
• project_name - str, name of objective of model
• raw_data_path - str, full file path of raw input data; assume parquet file
• input_col_nm - str, name of column which has text info to be classified
• targ_col_nm - str, name of column which has ground truth of text to be classified


# functionality
• reads in raw data at specified path (assume parquet)
• performs train test split on given input dataset per specifications in input args
• adds train, validation, test, and OOS column to reach row of given dataset
• creates directory struture of the form {base_dir}/{projects}/{project_name}/{project_[1,2,3,etc]}/split_data/ --> [data_config.json, split_data.parquet]
    • for example, if project_1 already exists in {base_dir}/{projects}/{project_name}/, it'll create a new dir for project_2
• writes two output files:
    • split_data.parquet: parquet of form [input_data, ground_truth_data, tts_label]
    • config file: json of all kwargs used as input arguments to this function


For example, new file path would be
/content/drive/MyDrive/CFG_complaints/projects/complaint_classification/project_1/split_data/ [config.json, output_parquet]

# Imports

In [ ]:
from google.colab import drive
import pandas as pd
import os
import json
import datetime

In [ ]:
# Mount Google Drive
### (when prompted, approve all permissions and continue)
drive.mount('/content/drive')

# Set base directory in google-drive for where to save files
base_dir = "/content/drive/MyDrive/CFG_complaints/"
### Files will be saved in form: {base_dir}/{projects}/{project_name}/{project_[1,2,3,etc]}/split_data/ --> [data_config.json, split_data.parquet]

# set path to existing raw data
raw_data_path = "/content/drive/MyDrive/CFG_complaints/raw_data/cfpb_complaints.parquet"

In [ ]:
def perform_train_test_split(
    tts_dict: dict,
    base_dir: str,
    project_name: str,
    raw_data_path: str,
    input_col_nm: str,
    targ_col_nm: str
) -> tuple:
    """
    Reads raw data, performs a train-test-validation split based on provided percentages
    (and an optional out-of-time period), adds a 'tts_label' column, creates a project-specific
    directory structure, and returns the split DataFrame and config. It prints the intended
    output paths but comments out the actual file writing.

    Args:
        tts_dict (dict): Dictionary containing train, validation, test percentages (floats)
                         and an optional 'out_of_time_period' (datetime object).
                         Example: {'train': 0.7, 'validation': 0.15, 'test': 0.15, 'out_of_time_period': datetime.date(2023, 1, 1)}
        base_dir (str): Base path to the projects directory where data will be saved.
                        E.g., '/content/drive/MyDrive/CFG_complaints/'
        project_name (str): Name of the objective of the model. E.g., 'complaint_classification'
        raw_data_path (str): Full file path of the raw input data (assumed to be a parquet file).
                             E.g., '/content/drive/MyDrive/CFG_complaints/raw_data/complaints.parquet'
        input_col_nm (str): Name of the column containing text information to be classified.
        targ_col_nm (str): Name of the column containing the ground truth for classification.

    Returns:
        tuple: A tuple containing:
            - pd.DataFrame: The DataFrame with the 'tts_label' column added.
            - dict: A dictionary containing all input arguments used for this function.

    Functionality:
    - Reads in raw data at the specified path (assumed parquet).
    - Performs train-validation-test split based on 'tts_dict' percentages.
    - If 'out_of_time_period' is provided in tts_dict, data before this date
      is split, and data on or after is labeled 'OOT'.
    - Adds 'train', 'validation', 'test', or 'OOT' label to each row in a new 'tts_label' column.
    - Creates a directory structure: `{base_dir}/projects/{project_name}/project_{N}/split_data/`
      where N is incremented if previous project directories exist.
    - Prints the intended paths for `split_data.parquet` and `data_config.json`.
    - The actual writing of files is commented out.
    """

    print(f"Reading raw data from: {raw_data_path}")
    df = pd.read_parquet(raw_data_path)

    # Ensure the target column is present
    if targ_col_nm not in df.columns:
        raise ValueError(f"Target column '{targ_col_nm}' not found in the DataFrame.")

    # Handle 'out_of_time_period' split first if specified
    out_of_time_period = tts_dict.get('out_of_time_period')
    if out_of_time_period and 'date' in df.columns:
        df['date'] = pd.to_datetime(df['date'])
        df_oot = df[df['date'] >= pd.to_datetime(out_of_time_period)].copy()
        df_oot['tts_label'] = 'OOT'
        df_split_candidates = df[df['date'] < pd.to_datetime(out_of_time_period)].copy()
    else:
        df_oot = pd.DataFrame()
        df_split_candidates = df.copy()

    # Perform train-validation-test split on the remaining data
    if not df_split_candidates.empty:
        # Ensure percentages sum close to 1
        total_percent = tts_dict['train'] + tts_dict['validation'] + tts_dict['test']
        if not (0.99 <= total_percent <= 1.01):
            print(f"Warning: Train/Validation/Test percentages sum to {total_percent}. Adjusting for consistency.")
            # Adjust to sum to 1 if slightly off
            df_split_candidates = df_split_candidates.sample(frac=1.0, random_state=42).reset_index(drop=True)
        else:
            df_split_candidates = df_split_candidates.sample(frac=total_percent, random_state=42).reset_index(drop=True)

        train_size = int(len(df_split_candidates) * tts_dict['train'])
        val_size = int(len(df_split_candidates) * tts_dict['validation'])

        df_train = df_split_candidates.iloc[:train_size].copy()
        df_val = df_split_candidates.iloc[train_size : train_size + val_size].copy()
        df_test = df_split_candidates.iloc[train_size + val_size:].copy()

        df_train['tts_label'] = 'train'
        df_val['tts_label'] = 'validation'
        df_test['tts_label'] = 'test'

        df_final_split = pd.concat([df_train, df_val, df_test, df_oot])
    else:
        df_final_split = df_oot # If only OOT data exists or no split candidates

    # Construct project directory path
    projects_base_path = os.path.join(base_dir, 'projects', project_name)
    project_instance_path = None
    for i in range(1, 100): # Try up to 99 projects
        temp_path = os.path.join(projects_base_path, f'project_{i}')
        if not os.path.exists(temp_path):
            project_instance_path = temp_path
            break
    if project_instance_path is None:
        raise RuntimeError(f"Could not find an available project_N directory under {projects_base_path}")

    output_dir = os.path.join(project_instance_path, 'split_data')

    # Create directory structure
    os.makedirs(output_dir, exist_ok=True)
    print(f"Directories Created At: {output_dir}")

    # Define output file paths
    output_parquet_path = os.path.join(output_dir, 'split_data.parquet')
    output_config_path = os.path.join(output_dir, 'data_config.json')

    # Prepare config data
    config_data = {
        'tts_dict': tts_dict,
        'base_dir': base_dir,
        'project_name': project_name,
        'raw_data_path': raw_data_path,
        'input_col_nm': input_col_nm,
        'targ_col_nm': targ_col_nm,
        'output_parquet_path': output_parquet_path,
        'output_config_path': output_config_path
    }

    # Rename columns
    df_final_split.rename(columns={input_col_nm: 'input_data', targ_col_nm: 'ground_truth_data'}, inplace=True)

    # Commented out actual file writing
    df_final_split.to_parquet(output_parquet_path, index=False)
    with open(output_config_path, 'w') as f:
        json.dump(config_data, f, indent=4, default=str) # Use default=str for datetime objects

    print(f"Split data written to: {output_parquet_path}")
    print(f"Configuration data written to: {output_config_path}")

In [ ]:
perform_train_test_split(
    tts_dict = {'train': 0.7, 'validation': 0.15, 'test': 0.15},
    base_dir = base_dir,
    project_name = "product_classification",
    raw_data_path = raw_data_path,
    input_col_nm = "complaint_what_happened",
    targ_col_nm = "product"
)